##Delta Live table
- Delta Live tables ot DLT is a framework for building relaible and maintainable data processing pipelines
- DLT simplifies the hard work of building large scale ETL while maintaining table dependencies and Data Quality
- DLT tables will always be preceeded by **LIVE** Keyword


<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%python
datasets_path = '/Volumes/demoworkspace/default/bookstore_dataset'

##Bronze Layer Tables

###orders_raw

In [0]:
%sql
CREATE OR REFRESH STREAMING LIVE TABLE orders_raw
COMMENT "The raw books orders, ingested from orders-raw"
AS SELECT * FROM cloud_files("${datasets.path}/orders-raw", "parquet",
                             map("Scheama","order_id STRING,order_timestamp LONG, customer_id STRING,quantity LONG"))

###customers

In [0]:
drop table if exists customers

In [0]:
%sql
create or refresh live table customers
comment "THe customers lookup table, ingested from  customers-json"
as
select * from json.`${datasets_path}/customers-json`

##Silver Layer Tables

###orders_cleaned

In [0]:
%sql
create or refresh streaming live table orders_cleaned(
    constraint valid_order_number expect (order_id IS NOT NULL) on violation drop row
)
comment "The cleaned books orders with vaild order_id"
as
    select order_id,quantity,o.customer_id,c.profile:first_name as f_name,c.profile:last_name as l_name,cast(from_unixtime(order_timestamp,'yyyy-MM-dd HH:mm:ss') as timestamp) as order_timestamp,c.profile:address:country as country
    from stream(live.order_raw) o
    left join live.customers c
    on o.customer_id = c.customer_id


- The **Constraint** Keyword enables DLT to collect metrics on constraint violations
- It provides an optional **ON VIOLATION** clause specifying an action to take on records that violate the constraints

>> Constraint violation

| **`ON VIOLATION`** | Behavior |
| --- | --- |
| **`DROP ROW`** | Discard records that violate constraints |
| **`FAIL UPDATE`** | Violated constraint causes the pipeline to fail  |
| Omitted | Records violating constraints will be kept, and reported in metrics |

##Gold Tables

In [0]:
%sql
create or refresh live table cn_daily_customer_books
comment "Daily number of books per customer in China"
as
    select customer_id,f_name,l_name,date_trunc("DD", order_timestamp) as order_date, sum(quantity) as books_counts
    from live.orders_cleaned
    where country = "China"
    group by customer_id,f_name,l_name,date_trunc("DD", order_timestamp)